# Medical Robotics Project 

1. **YOLO + Depth Anything V2**
2. **U-Net + Depth Anything V2**

The dataset was exported in two architecture-specific formats. Both formats contain the same annotated frames and share the same train/validation/test split. The YOLO representation stores polygon annotations, whereas the U-Net representation stores raster masks and preview images. The two representations are complementary views of the same underlying dataset.

```text
YOLO dataset
    ├── images
    ├── labels       → tool segmentation
    └── tti_labels   → TTI segmentation/detection

U-Net dataset
    ├── images
    ├── masks        → tool segmentation
    ├── tti_masks    → TTI/contact ground truth
    ├── preview_masks
    └── preview_tti_masks
```


| Informazione            | Dataset YOLO                 | Dataset U-Net                   |
| ----------------------- | ---------------------------- | ------------------------------- |
| Immagini                | Sì                           | Sì                              |
| Split train/val/test    | Sì                           | Sì                              |
| Tool polygons           | Sì, in labels                | No, rappresentati come mask     |
| TTI polygons            | Sì, in tti_labels            | No, rappresentati come TTI mask |
| Tool raster masks       | No o non necessarie          | Sì, in masks/mask               |
| TTI raster masks        | No o non necessarie          | Sì, in tti_masks                |
| Preview colorate        | Non necessarie               | Sì, solo controllo visivo       |
| Addestramento YOLO      | Diretto                      | Non diretto                     |
| Addestramento U-Net     | Non diretto                  | Diretto                         |
| Valutazione contact/TTI | Label poligonali disponibili | TTI mask disponibili            |


Per confronto:

```text
YOLO tool mask ─────┐
                    ├── Depth Anything v2 ── contact isolation
U-Net tool mask ────┘

```



@misc{jocher2026ultralyticsyolo26unifiedrealtime,
  title = {Ultralytics YOLO26: Unified Real-Time End-to-End Vision Models},
  author = {Glenn Jocher and Jing Qiu and Mengyu Liu and Shuai Lyu and Fatih Cagatay Akyon and Muhammet Esat Kalfaoglu},
  year = {2026},
  eprint = {2606.03748},
  archivePrefix = {arXiv},
  primaryClass = {cs.CV},
  doi = {10.48550/arXiv.2606.03748},
  url = {https://arxiv.org/abs/2606.03748},
}

### .

In [ ]:
from pathlib import Path

import pandas as pd


YOLO_ROOT = Path("yolo_dataset")

# Se DATAROOT esiste già nel notebook U-Net, usalo.
# Altrimenti modifica il percorso manualmente.
UNET_ROOT = (
    Path("unet_dataset"))


def get_image_files(directory):
    directory = Path(directory)

    if not directory.exists():
        return []

    image_files = []

    for pattern in [
        "*.png",
        "*.jpg",
        "*.jpeg",
        "*.bmp",
        "*.tif",
        "*.tiff",
    ]:
        image_files.extend(
            directory.glob(pattern)
        )

    return sorted(image_files)


def get_image_stems(directory):
    return {
        path.stem
        for path in get_image_files(directory)
    }

In [ ]:
def compare_image_splits(
    yolo_root,
    unet_root,
):
    yolo_root = Path(yolo_root)
    unet_root = Path(unet_root)

    for split in [
        "train",
        "val",
        "test",
    ]:
        yolo_stems = get_image_stems(
            yolo_root / "images" / split
        )

        unet_stems = get_image_stems(
            unet_root / "images" / split
        )

        only_yolo = sorted(
            yolo_stems - unet_stems
        )

        only_unet = sorted(
            unet_stems - yolo_stems
        )

        print(f"\n{split.upper()}")
        print("YOLO:", len(yolo_stems))
        print("U-Net:", len(unet_stems))
        print("Solo YOLO:", len(only_yolo))
        print("Solo U-Net:", len(only_unet))

        if only_yolo:
            print(
                "Esempi solo YOLO:",
                only_yolo[:5],
            )

        if only_unet:
            print(
                "Esempi solo U-Net:",
                only_unet[:5],
            )


compare_image_splits(
    yolo_root=YOLO_ROOT,
    unet_root=UNET_ROOT,
)

In [ ]:
def check_binary_yolo_labels(label_root):
    label_root = Path(label_root)

    if not label_root.exists():
        raise FileNotFoundError(
            f"Cartella non trovata: "
            f"{label_root.resolve()}"
        )

    class_ids = set()
    invalid_lines = []

    for label_path in label_root.rglob("*.txt"):
        with open(
            label_path,
            "r",
            encoding="utf-8",
        ) as file:
            for line_number, line in enumerate(
                file,
                start=1,
            ):
                values = line.strip().split()

                if not values:
                    continue

                try:
                    class_id = int(values[0])
                except ValueError:
                    invalid_lines.append(
                        (
                            label_path,
                            line_number,
                            line,
                        )
                    )
                    continue

                class_ids.add(class_id)

                if class_id != 0:
                    invalid_lines.append(
                        (
                            label_path,
                            line_number,
                            line,
                        )
                    )

    print("Classi trovate:", sorted(class_ids))
    print(
        "Righe non binarie:",
        len(invalid_lines),
    )

    if invalid_lines:
        print(
            "Primo errore:",
            invalid_lines[0],
        )
        raise ValueError(
            "Le label contengono classi diverse da 0."
        )


check_binary_yolo_labels(
    YOLO_ROOT / "labels"
)

## U-Net

### Import

In [ ]:
import random
import shutil
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

### Dataset

In [ ]:
# Percorsi relativi alla cartella da cui è aperto il notebook
DATA_ROOT = Path("unet_dataset")
CHECKPOINT_PATH = Path("checkpoints/unet_best.pth")
OUTPUT_DIR = Path("outputs/unet")

# Parametri
IMAGE_SIZE = 512
BATCH_SIZE = 2
NUM_WORKERS = 2
EPOCHS = 30
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5
THRESHOLD = 0.5

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

CHECKPOINT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Working directory:", Path.cwd())
print("Dataset root:", DATA_ROOT.resolve())
print("Dataset exists:", DATA_ROOT.exists())
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nCartelle dataset:")
print("images/train:", (DATA_ROOT / "images" / "train").exists())
print("images/val:", (DATA_ROOT / "images" / "val").exists())
print("images/test:", (DATA_ROOT / "images" / "test").exists())
print("masks/train:", (DATA_ROOT / "masks" / "train").exists())
print("masks/val:", (DATA_ROOT / "masks" / "val").exists())
print("masks/test:", (DATA_ROOT / "masks" / "test").exists())

In [ ]:
print("Contenuto di unet_dataset:")
for path in sorted(DATA_ROOT.iterdir()):
    print(path)

print("\nSottocartelle di images:")
images_root = DATA_ROOT / "images"

if images_root.exists():
    for path in sorted(images_root.iterdir()):
        print(path, "directory:", path.is_dir())

print("\nSottocartelle di masks:")
masks_root = DATA_ROOT / "masks"

if masks_root.exists():
    for path in sorted(masks_root.iterdir()):
        print(path, "directory:", path.is_dir())

print("\nNumero di file per cartella:")
for root in [images_root, masks_root]:
    if root.exists():
        for split_dir in sorted(root.iterdir()):
            if split_dir.is_dir():
                files = [
                    p for p in split_dir.iterdir()
                    if p.is_file()
                ]
                print(f"{split_dir}: {len(files)} file")

some file haven't the relative mask, so we check the images without mask

In [ ]:
images_train_dir = DATA_ROOT / "images" / "train"
masks_train_dir = DATA_ROOT / "masks" / "train"

image_paths = sorted(
    list(images_train_dir.glob("*.png")) +
    list(images_train_dir.glob("*.jpg")) +
    list(images_train_dir.glob("*.jpeg"))
)

mask_paths = sorted(
    list(masks_train_dir.glob("*.png")) +
    list(masks_train_dir.glob("*.jpg")) +
    list(masks_train_dir.glob("*.jpeg"))
)

image_stems = {path.stem for path in image_paths}
mask_stems = {path.stem for path in mask_paths}

images_without_masks = sorted(image_stems - mask_stems)
masks_without_images = sorted(mask_stems - image_stems)

print("Numero immagini:", len(image_paths))
print("Numero maschere:", len(mask_paths))

print("\nImmagini senza maschera:")
print(images_without_masks)
print("Totale:", len(images_without_masks))

print("\nMaschere senza immagine:")
print(masks_without_images)
print("Totale:", len(masks_without_images))

create new dataset structure:


In [ ]:
'''
unet_dataset_split/
├── images/
│   ├── train/
│   ├── val/
│   └── test/
└── masks/
    ├── train/
    ├── val/
    └── test/
'''

In [ ]:
source_images_dir = DATA_ROOT.parent / "unet_dataset" / "images" / "train"
source_masks_dir = DATA_ROOT.parent / "unet_dataset" / "masks" / "train"

split_root = Path("unet_dataset_split")

if split_root.exists():
    shutil.rmtree(split_root)

for split in ["train", "val", "test"]:
    (
        split_root /
        "images" /
        split
    ).mkdir(parents=True, exist_ok=True)

    (
        split_root /
        "masks" /
        split
    ).mkdir(parents=True, exist_ok=True)

image_paths = sorted(
    list(source_images_dir.glob("*.png")) +
    list(source_images_dir.glob("*.jpg")) +
    list(source_images_dir.glob("*.jpeg"))
)

mask_paths = sorted(
    list(source_masks_dir.glob("*.png")) +
    list(source_masks_dir.glob("*.jpg")) +
    list(source_masks_dir.glob("*.jpeg"))
)

mask_by_stem = {
    mask_path.stem: mask_path
    for mask_path in mask_paths
}

valid_pairs = [
    (image_path, mask_by_stem[image_path.stem])
    for image_path in image_paths
    if image_path.stem in mask_by_stem
]

rng = random.Random(42)
rng.shuffle(valid_pairs)

n_total = len(valid_pairs)
n_train = int(0.70 * n_total)
n_val = int(0.20 * n_total)

split_pairs = {
    "train": valid_pairs[:n_train],
    "val": valid_pairs[n_train:n_train + n_val],
    "test": valid_pairs[n_train + n_val:]
}

for split, pairs in split_pairs.items():
    for image_path, mask_path in pairs:
        shutil.copy2(
            image_path,
            split_root / "images" / split / image_path.name
        )

        shutil.copy2(
            mask_path,
            split_root / "masks" / split / mask_path.name
        )

print("Nuovi split creati:")
for split, pairs in split_pairs.items():
    print(f"{split}: {len(pairs)} coppie")

print(
    "\nPercorso:",
    split_root.resolve()
)

In [ ]:
for split in ["train", "val", "test"]:
    images_dir = split_root / "images" / split
    masks_dir = split_root / "masks" / split

    images = list(images_dir.iterdir())
    masks = list(masks_dir.iterdir())

    image_stems = {p.stem for p in images}
    mask_stems = {p.stem for p in masks}

    print(f"\n{split.upper()}")
    print("Immagini:", len(images))
    print("Maschere:", len(masks))
    print("Immagini senza maschera:", image_stems - mask_stems)
    print("Maschere senza immagine:", mask_stems - image_stems)

### U-Net class

In [ ]:
DATA_ROOT = split_root

print("Nuovo dataset root:")
print(DATA_ROOT.resolve())

if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"Dataset split non trovato: {DATA_ROOT.resolve()}"
    )

for split in ["train", "val", "test"]:
    images_dir = DATA_ROOT / "images" / split
    masks_dir = DATA_ROOT / "masks" / split

    image_count = len(list(images_dir.glob("*"))) if images_dir.exists() else 0
    mask_count = len(list(masks_dir.glob("*"))) if masks_dir.exists() else 0

    print(
        f"{split}: "
        f"{image_count} immagini, "
        f"{mask_count} maschere"
    )

In [ ]:
class UNetDataset(Dataset):
    def __init__(
        self,
        root,
        split,
        image_size=512,
        augment=False
    ):
        self.root = Path(root)
        self.split = split
        self.image_size = image_size
        self.augment = augment

        self.image_dir = self.root / "images" / split
        self.mask_dir = self.root / "masks" / split

        if not self.image_dir.exists():
            raise FileNotFoundError(
                f"Cartella immagini non trovata: "
                f"{self.image_dir.resolve()}"
            )

        if not self.mask_dir.exists():
            raise FileNotFoundError(
                f"Cartella maschere non trovata: "
                f"{self.mask_dir.resolve()}"
            )

        image_paths = sorted(
            list(self.image_dir.glob("*.png")) +
            list(self.image_dir.glob("*.jpg")) +
            list(self.image_dir.glob("*.jpeg"))
        )

        mask_paths = sorted(
            list(self.mask_dir.glob("*.png")) +
            list(self.mask_dir.glob("*.jpg")) +
            list(self.mask_dir.glob("*.jpeg"))
        )

        mask_by_stem = {
            mask_path.stem: mask_path
            for mask_path in mask_paths
        }

        self.samples = [
            (image_path, mask_by_stem[image_path.stem])
            for image_path in image_paths
            if image_path.stem in mask_by_stem
        ]

        if len(self.samples) == 0:
            raise RuntimeError(
                f"Nessuna coppia immagine-maschera valida in "
                f"{self.image_dir.resolve()}"
            )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_path, mask_path = self.samples[idx]

        image_bgr = cv2.imread(
            str(image_path),
            cv2.IMREAD_COLOR
        )

        mask = cv2.imread(
            str(mask_path),
            cv2.IMREAD_UNCHANGED
        )

        if image_bgr is None:
            raise RuntimeError(
                f"Immagine non leggibile: {image_path}"
            )

        if mask is None:
            raise RuntimeError(
                f"Maschera non leggibile: {mask_path}"
            )

        if mask.ndim == 3:
            mask = mask[:, :, 0]

        image_bgr = cv2.resize(
            image_bgr,
            (self.image_size, self.image_size),
            interpolation=cv2.INTER_LINEAR
        )

        mask = cv2.resize(
            mask,
            (self.image_size, self.image_size),
            interpolation=cv2.INTER_NEAREST
        )

        if self.augment:
            if random.random() < 0.5:
                image_bgr = np.fliplr(image_bgr).copy()
                mask = np.fliplr(mask).copy()

            if random.random() < 0.5:
                image_bgr = np.flipud(image_bgr).copy()
                mask = np.flipud(mask).copy()

        image_rgb = cv2.cvtColor(
            image_bgr,
            cv2.COLOR_BGR2RGB
        )

        image_rgb = image_rgb.astype(np.float32) / 255.0
        image_rgb = np.transpose(
            image_rgb,
            (2, 0, 1)
        )

        binary_mask = (mask > 0).astype(np.float32)
        binary_mask = np.expand_dims(
            binary_mask,
            axis=0
        )

        return {
            "image": torch.from_numpy(image_rgb).float(),
            "mask": torch.from_numpy(binary_mask).float(),
            "id": image_path.stem
        }

### Dataset and DataLoader

In [ ]:
train_dataset = UNetDataset(
    root=DATA_ROOT,
    split="train",
    image_size=IMAGE_SIZE,
    augment=True
)

val_dataset = UNetDataset(
    root=DATA_ROOT,
    split="val",
    image_size=IMAGE_SIZE,
    augment=False
)

test_dataset = UNetDataset(
    root=DATA_ROOT,
    split="test",
    image_size=IMAGE_SIZE,
    augment=False
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

print("Dataset e DataLoader creati correttamente.")
print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))
print("\nTrain batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

In [ ]:
batch = next(iter(train_loader))

images = batch["image"]
masks = batch["mask"]

print("Images shape:", images.shape)
print("Masks shape:", masks.shape)

print("Images dtype:", images.dtype)
print("Masks dtype:", masks.dtype)

print("Images min/max:", images.min().item(), images.max().item())
print("Masks unique values:", torch.unique(masks))
print("IDs:", batch["id"])

### U-Net

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                in_ch,
                out_ch,
                kernel_size=3,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_ch,
                out_ch,
                kernel_size=3,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class UNetSmall(nn.Module):
    def __init__(
        self,
        in_channels=3,
        out_channels=1,
        base_channels=32
    ):
        super().__init__()

        # Encoder
        self.enc1 = DoubleConv(
            in_channels,
            base_channels
        )
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = DoubleConv(
            base_channels,
            base_channels * 2
        )
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = DoubleConv(
            base_channels * 2,
            base_channels * 4
        )
        self.pool3 = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = DoubleConv(
            base_channels * 4,
            base_channels * 8
        )

        # Decoder
        self.up3 = nn.ConvTranspose2d(
            base_channels * 8,
            base_channels * 4,
            kernel_size=2,
            stride=2
        )

        self.dec3 = DoubleConv(
            base_channels * 8,
            base_channels * 4
        )

        self.up2 = nn.ConvTranspose2d(
            base_channels * 4,
            base_channels * 2,
            kernel_size=2,
            stride=2
        )

        self.dec2 = DoubleConv(
            base_channels * 4,
            base_channels * 2
        )

        self.up1 = nn.ConvTranspose2d(
            base_channels * 2,
            base_channels,
            kernel_size=2,
            stride=2
        )

        self.dec1 = DoubleConv(
            base_channels * 2,
            base_channels
        )

        # Output binario
        self.head = nn.Conv2d(
            base_channels,
            out_channels,
            kernel_size=1
        )

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))

        # Bottleneck
        b = self.bottleneck(self.pool3(e3))

        # Decoder
        d3 = self.up3(b)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.head(d1)

### Loss and metrics

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)

        probs = probs.flatten(1)
        targets = targets.flatten(1)

        intersection = (probs * targets).sum(dim=1)

        dice = (
            2.0 * intersection + self.smooth
        ) / (
            probs.sum(dim=1) +
            targets.sum(dim=1) +
            self.smooth
        )

        return 1.0 - dice.mean()


class BCEDiceLoss(nn.Module):
    def __init__(
        self,
        bce_weight=0.5,
        dice_weight=0.5
    ):
        super().__init__()

        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()

        self.bce_weight = bce_weight
        self.dice_weight = dice_weight

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        dice_loss = self.dice(logits, targets)

        return (
            self.bce_weight * bce_loss +
            self.dice_weight * dice_loss
        )


def segmentation_metrics(
    logits,
    targets,
    threshold=0.5,
    eps=1e-7
):
    probs = torch.sigmoid(logits)
    predictions = (probs >= threshold).float()

    intersection = (
        predictions * targets
    ).sum(dim=(1, 2, 3))

    union = (
        (predictions + targets) > 0
    ).float().sum(dim=(1, 2, 3))

    dice = (
        2.0 * intersection + eps
    ) / (
        predictions.sum(dim=(1, 2, 3)) +
        targets.sum(dim=(1, 2, 3)) +
        eps
    )

    iou = (
        intersection + eps
    ) / (
        union + eps
    )

    return iou.mean().item(), dice.mean().item()

### Training

In [ ]:
def run_epoch(
    model,
    loader,
    criterion,
    device,
    optimizer=None,
    threshold=0.5
):
    is_training = optimizer is not None

    if is_training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_iou = 0.0
    total_dice = 0.0

    for batch in loader:
        batch_images = batch["image"].to(
            device,
            non_blocking=True
        )

        batch_masks = batch["mask"].to(
            device,
            non_blocking=True
        )

        if is_training:
            optimizer.zero_grad(
                set_to_none=True
            )

        with torch.set_grad_enabled(is_training):
            logits = model(batch_images)

            loss = criterion(
                logits,
                batch_masks
            )

            if is_training:
                loss.backward()
                optimizer.step()

        batch_iou, batch_dice = segmentation_metrics(
            logits.detach(),
            batch_masks,
            threshold=threshold
        )

        total_loss += loss.item()
        total_iou += batch_iou
        total_dice += batch_dice

    num_batches = len(loader)

    return {
        "loss": total_loss / num_batches,
        "iou": total_iou / num_batches,
        "dice": total_dice / num_batches
    }

In [ ]:
model = UNetSmall(
    in_channels=3,
    out_channels=1,
    base_channels=32
).to(DEVICE)

criterion = BCEDiceLoss(
    bce_weight=0.5,
    dice_weight=0.5
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3
)

num_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("Modello inizializzato.")
print("Device:", next(model.parameters()).device)
print("Trainable parameters:", num_parameters)
print("Learning rate:", optimizer.param_groups[0]["lr"])

In [ ]:
# Variabili di monitoraggio
best_val_iou = -1.0
history = []

print("Training U-Net")
print("Epoche:", EPOCHS)
print("Device:", DEVICE)
print("Checkpoint:", CHECKPOINT_PATH)

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(
        model=model,
        loader=train_loader,
        criterion=criterion,
        device=DEVICE,
        optimizer=optimizer,
        threshold=THRESHOLD
    )

    with torch.no_grad():
        val_metrics = run_epoch(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=DEVICE,
            optimizer=None,
            threshold=THRESHOLD
        )

    scheduler.step(val_metrics["iou"])

    current_lr = optimizer.param_groups[0]["lr"]

    epoch_result = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_iou": train_metrics["iou"],
        "train_dice": train_metrics["dice"],
        "val_loss": val_metrics["loss"],
        "val_iou": val_metrics["iou"],
        "val_dice": val_metrics["dice"],
        "lr": current_lr
    }

    history.append(epoch_result)

    print(
        f"Epoch {epoch:03d}/{EPOCHS} | "
        f"train loss: {train_metrics['loss']:.4f} | "
        f"train IoU: {train_metrics['iou']:.4f} | "
        f"train Dice: {train_metrics['dice']:.4f} | "
        f"val loss: {val_metrics['loss']:.4f} | "
        f"val IoU: {val_metrics['iou']:.4f} | "
        f"val Dice: {val_metrics['dice']:.4f} | "
        f"lr: {current_lr:.2e}"
    )

    if val_metrics["iou"] > best_val_iou:
        best_val_iou = val_metrics["iou"]

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_iou": val_metrics["iou"],
                "val_dice": val_metrics["dice"],
                "threshold": THRESHOLD
            },
            CHECKPOINT_PATH
        )

        print(
            "  Best checkpoint salvato:",
            CHECKPOINT_PATH
        )

print("\nTraining completato.")
print(f"Best validation IoU: {best_val_iou:.4f}")
print("Checkpoint:", CHECKPOINT_PATH.resolve())

In [ ]:
history_df = pd.DataFrame(history)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 5)
)

# Loss
axes[0].plot(
    history_df["epoch"],
    history_df["train_loss"],
    label="Train"
)

axes[0].plot(
    history_df["epoch"],
    history_df["val_loss"],
    label="Validation"
)

axes[0].set_title("U-Net Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(alpha=0.3)
axes[0].legend()

# IoU
axes[1].plot(
    history_df["epoch"],
    history_df["train_iou"],
    label="Train"
)

axes[1].plot(
    history_df["epoch"],
    history_df["val_iou"],
    label="Validation"
)

axes[1].set_title("U-Net IoU")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("IoU")
axes[1].set_ylim(0, 1)
axes[1].grid(alpha=0.3)
axes[1].legend()

# Dice
axes[2].plot(
    history_df["epoch"],
    history_df["train_dice"],
    label="Train"
)

axes[2].plot(
    history_df["epoch"],
    history_df["val_dice"],
    label="Validation"
)

axes[2].set_title("U-Net Dice")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Dice")
axes[2].set_ylim(0, 1)
axes[2].grid(alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
best_model = UNetSmall(
    in_channels=3,
    out_channels=1,
    base_channels=32
).to(DEVICE)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE
)

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    best_model.load_state_dict(checkpoint["model_state_dict"])
else:
    best_model.load_state_dict(checkpoint)

best_model.eval()

print("Best checkpoint caricato correttamente.")
print("Checkpoint:", CHECKPOINT_PATH.resolve())

### Results

In [ ]:
test_metrics = run_epoch(
    model=best_model,
    loader=test_loader,
    criterion=criterion,
    device=DEVICE,
    optimizer=None,
    threshold=THRESHOLD
)

print("Risultati sul test set:")
print(f"Loss: {test_metrics['loss']:.4f}")
print(f"IoU:  {test_metrics['iou']:.4f}")
print(f"Dice: {test_metrics['dice']:.4f}")

In [ ]:
@torch.no_grad()
def show_prediction(
    model,
    dataset,
    index=0,
    threshold=0.5
):
    model.eval()

    sample = dataset[index]

    image = sample["image"].unsqueeze(0).to(DEVICE)
    ground_truth = sample["mask"][0].cpu().numpy()

    logits = model(image)

    probability = torch.sigmoid(
        logits[0, 0]
    ).cpu().numpy()

    prediction = (
        probability >= threshold
    ).astype(np.uint8)

    image_rgb = sample["image"].permute(
        1,
        2,
        0
    ).cpu().numpy()

    fig, axes = plt.subplots(
        1,
        5,
        figsize=(25, 5)
    )

    axes[0].imshow(image_rgb)
    axes[0].set_title("Input")

    axes[1].imshow(
        ground_truth,
        cmap="gray",
        vmin=0,
        vmax=1
    )
    axes[1].set_title("Ground truth")

    axes[2].imshow(
        probability,
        cmap="viridis",
        vmin=0,
        vmax=1
    )
    axes[2].set_title("Probability map")

    axes[3].imshow(
        prediction,
        cmap="gray",
        vmin=0,
        vmax=1
    )
    axes[3].set_title("U-Net prediction")

    axes[4].imshow(image_rgb)
    axes[4].imshow(
        prediction,
        cmap="Reds",
        alpha=0.45,
        vmin=0,
        vmax=1
    )
    axes[4].set_title("Prediction overlay")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

    print("Sample ID:", sample["id"])
    print(
        "Ground-truth area:",
        float(ground_truth.mean())
    )
    print(
        "Predicted area:",
        float(prediction.mean())
    )


show_prediction(
    model=best_model,
    dataset=test_dataset,
    index=0,
    threshold=THRESHOLD
)

## YOLO

### 1. Import e funzioni

In [1]:
from pathlib import Path
import time

import cv2
import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt

from ultralytics import YOLO

In [2]:
def ensure_dir(path):
    Path(path).mkdir(
        parents=True,
        exist_ok=True,
    )


def resolve_device(device=None):
    if device is not None:
        return device

    return 0 if torch.cuda.is_available() else "cpu"


def get_image_files(directory):
    directory = Path(directory)

    image_paths = []

    for pattern in [
        "*.jpg",
        "*.jpeg",
        "*.png",
        "*.bmp",
        "*.tif",
        "*.tiff",
    ]:
        image_paths.extend(
            directory.glob(pattern)
        )

    return sorted(image_paths)


def save_image(path, image):
    path = Path(path)

    ensure_dir(path.parent)

    if not cv2.imwrite(
        str(path),
        image,
    ):
        raise IOError(
            f"Impossibile salvare: {path}"
        )


def overlay_mask(
    image_bgr,
    mask,
    color=(0, 255, 255),
    alpha=0.45,
):
    colored = image_bgr.copy()
    colored[mask > 0] = color

    return cv2.addWeighted(
        colored,
        alpha,
        image_bgr,
        1.0 - alpha,
        0,
    )

### config di base


In [3]:
cfg = {
    "device": None,

    "dataset_root": "yolo_dataset",

    "data_yaml": (
        "configs/data_tools_yolo26.yaml"
    ),

    # YOLO26 per INSTANCE SEGMENTATION.
    # Non usare yolo26n.pt: produce box, non mask.
    "base_weights": "yolo26n-seg.pt",

    "image_size": 512,
    "batch_size": 4,
    "epochs": 150,
    "patience": 20,

    "project": "outputs",
    "run_name": "yolo26n_tools",

    # Solo per l'inferenza manuale.
    "confidence": 0.60,
}

YOLO_ROOT = Path(
    cfg["dataset_root"]
)

YOLO_DATA_YAML = Path(
    cfg["data_yaml"]
)



YOLO_OUTPUT_DIR = Path(
    cfg["project"]
)

YOLO_DEVICE = resolve_device(
    cfg["device"]
)

ensure_dir(
    YOLO_DATA_YAML.parent
)

ensure_dir(
    YOLO_OUTPUT_DIR
)

print("Dataset:", YOLO_ROOT.resolve())
print("Data YAML:", YOLO_DATA_YAML.resolve())
print("Device:", YOLO_DEVICE)
print("Image size:", cfg["image_size"])
print("Batch size:", cfg["batch_size"])

Dataset: /home/federico-ai-rob/Scrivania/AIRO/MR_project/yolo_dataset
Data YAML: /home/federico-ai-rob/Scrivania/AIRO/MR_project/configs/data_tools_yolo26.yaml
Device: 0
Image size: 512
Batch size: 4


### Dataset YOLO

In [4]:
class_names = {
    0: "tool_0",
    1: "tool_1",
    2: "tool_2",
    3: "tool_3",
    4: "tool_4",
    5: "tool_5",
    6: "tool_6",
    7: "tool_7",
    8: "tool_8",
    9: "tool_9",
    10: "tool_10_UNUSED",
    11: "tool_11",
}

data_config = {
    "path": str(
        YOLO_ROOT.resolve()
    ),

    "train": str(
        (
            YOLO_ROOT
            / "images"
            / "train"
        ).resolve()
    ),

    "val": str(
        (
            YOLO_ROOT
            / "images"
            / "val"
        ).resolve()
    ),

    "test": str(
        (
            YOLO_ROOT
            / "images"
            / "test"
        ).resolve()
    ),

    "names": class_names,
}

with open(
    YOLO_DATA_YAML,
    "w",
    encoding="utf-8",
) as file:
    yaml.safe_dump(
        data_config,
        file,
        sort_keys=False,
        allow_unicode=True,
    )

print(
    "YAML creato:",
    YOLO_DATA_YAML.resolve(),
)

with open(
    YOLO_DATA_YAML,
    "r",
    encoding="utf-8",
) as file:
    print(file.read())

YAML creato: /home/federico-ai-rob/Scrivania/AIRO/MR_project/configs/data_tools_yolo26.yaml
path: /home/federico-ai-rob/Scrivania/AIRO/MR_project/yolo_dataset
train: /home/federico-ai-rob/Scrivania/AIRO/MR_project/yolo_dataset/images/train
val: /home/federico-ai-rob/Scrivania/AIRO/MR_project/yolo_dataset/images/val
test: /home/federico-ai-rob/Scrivania/AIRO/MR_project/yolo_dataset/images/test
names:
  0: tool_0
  1: tool_1
  2: tool_2
  3: tool_3
  4: tool_4
  5: tool_5
  6: tool_6
  7: tool_7
  8: tool_8
  9: tool_9
  10: tool_10_UNUSED
  11: tool_11



In [5]:
for split in ["train", "val", "test"]:
    image_dir = YOLO_ROOT / "images" / split
    label_dir = YOLO_ROOT / "labels" / split

    image_count = len(
        get_image_files(image_dir)
    )

    label_count = len(
        list(label_dir.glob("*.txt"))
    )

    print(
        f"{split}: "
        f"{image_count} immagini, "
        f"{label_count} label"
    )

train: 2880 immagini, 2880 label
val: 760 immagini, 760 label
test: 182 immagini, 182 label


### YOLO stage

In [6]:
# ============================================================
# YOLO26: training, validation e inferenza
# ============================================================

def predict_instances(
    model,
    image_bgr,
    confidence,
    device,
):
    """
    Esegue YOLO26-seg su una singola immagine e restituisce
    una lista di istanze predette.

    Ogni istanza contiene:
    - mask binaria a risoluzione originale;
    - class_id;
    - confidence.
    """
    results = model.predict(
        source=image_bgr,
        conf=confidence,
        device=device,
        verbose=False,
    )

    result = results[0]
    height, width = image_bgr.shape[:2]

    if (
        result.masks is None
        or result.boxes is None
    ):
        return []

    masks = (
        result.masks.data
        .detach()
        .cpu()
        .numpy()
    )

    class_ids = (
        result.boxes.cls
        .detach()
        .cpu()
        .numpy()
        .astype(np.int64)
    )

    confidences = (
        result.boxes.conf
        .detach()
        .cpu()
        .numpy()
    )

    instances = []

    for mask, class_id, conf in zip(
        masks,
        class_ids,
        confidences,
    ):
        binary_mask = (
            mask > 0.5
        ).astype(np.uint8)

        binary_mask = cv2.resize(
            binary_mask,
            (width, height),
            interpolation=cv2.INTER_NEAREST,
        )

        instances.append({
            "mask": binary_mask,
            "class_id": int(class_id),
            "confidence": float(conf),
        })

    return instances


def merge_tool_masks(
    instances,
    image_shape,
):
    """
    Aggrega le mask di tutte le istanze/classi.

    Output:
    - 0: background
    - 255: pixel appartenente a uno strumento
    """
    height, width = image_shape[:2]

    tool_mask = np.zeros(
        (height, width),
        dtype=np.uint8,
    )

    for instance in instances:
        tool_mask[
            instance["mask"] > 0
        ] = 255

    return tool_mask

### Training — modello strumenti

In [7]:
model = YOLO(
    cfg["base_weights"]
)

start_time = time.time()

train_results = model.train(
    data=str(YOLO_DATA_YAML),
    task="segment",

    epochs=cfg["epochs"],
    imgsz=cfg["image_size"],
    batch=cfg["batch_size"],
    patience=cfg["patience"],

    device=YOLO_DEVICE,

    project=str(YOLO_OUTPUT_DIR),
    name=cfg["run_name"],

    # Il run viene sovrascritto se esiste.
    exist_ok=True,
)

elapsed_minutes = (
    time.time() - start_time
) / 60.0

print(
    f"Training completato in "
    f"{elapsed_minutes:.2f} minuti."
)

New https://pypi.org/project/ultralytics/8.4.120 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.68 🚀 Python-3.12.3 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 7708MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=configs/data_tools_yolo26.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-seg.pt, momentum=0.937, mosaic=1.

### Inferenza di verifica

Carica i pesi migliori appena addestrati e verifica visivamente la maschera predetta su un'immagine di test.


In [9]:
best_weights = Path(
    "runs/segment/outputs/"
    "yolo26n_tools/weights/best.pt"
)

if not best_weights.exists():
    raise FileNotFoundError(
        "best.pt non trovato: "
        f"{best_weights.resolve()}"
    )

best_model = YOLO(
    str(best_weights)
)

print(
    "Best checkpoint caricato:",
    best_weights.resolve()
)

Best checkpoint caricato: /home/federico-ai-rob/Scrivania/AIRO/MR_project/runs/segment/outputs/yolo26n_tools/weights/best.pt


In [10]:
validation_results = best_model.val(
    data=str(YOLO_DATA_YAML),
    split="val",
    device=YOLO_DEVICE,
)

print(
    "Validation completata."
)

Ultralytics 8.4.68 🚀 Python-3.12.3 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 7708MiB)
YOLO26n-seg summary (fused): 139 layers, 2,691,224 parameters, 0 gradients, 9.0 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3210.3±1843.4 MB/s, size: 660.1 KB)
val: Scanning /home/federico-ai-rob/Scrivania/AIRO/MR_project/yolo_dataset/labels/val.cache... 760 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 760/760 455.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 48/48 10.5it/s 4.6s0.1s
                   all        760       1157       0.74      0.646      0.687      0.578       0.74      0.646      0.684      0.538
                tool_0         15         18          0          0    0.00399    0.00359          0          0    0.00399    0.00334
                tool_1        139        139      0.818      0.856      0.837      0.713      0.818    

In [11]:
test_images_dir = (
    YOLO_ROOT
    / "images"
    / "test"
)

test_image_paths = get_image_files(
    test_images_dir
)

if not test_image_paths:
    raise RuntimeError(
        "Nessuna immagine trovata nel test set."
    )

sample_path = test_image_paths[0]

image_bgr = cv2.imread(
    str(sample_path)
)

if image_bgr is None:
    raise RuntimeError(
        f"Immagine non leggibile: "
        f"{sample_path}"
    )

instances = predict_instances(
    model=best_model,
    image_bgr=image_bgr,
    confidence=cfg["confidence"],
    device=YOLO_DEVICE,
)

tool_mask = merge_tool_masks(
    instances=instances,
    image_shape=image_bgr.shape,
)

overlay = overlay_mask(
    image_bgr=image_bgr,
    mask=tool_mask,
)

print(
    "Numero istanze predette:",
    len(instances),
)

print(
    "Classi predette:",
    [
        item["class_id"]
        for item in instances
    ],
)

print(
    "Confidence:",
    [
        round(
            item["confidence"],
            3,
        )
        for item in instances
    ],
)

Numero istanze predette: 1
Classi predette: [1]
Confidence: [0.903]


In [12]:
inference_dir = (
    YOLO_OUTPUT_DIR
    / cfg["run_name"]
    / "inference_example"
)

save_image(
    inference_dir / "input.png",
    image_bgr,
)

save_image(
    inference_dir / "tool_mask.png",
    tool_mask,
)

save_image(
    inference_dir / "overlay.png",
    overlay,
)

print(
    "Output salvati in:",
    inference_dir.resolve(),
)

Output salvati in: /home/federico-ai-rob/Scrivania/AIRO/MR_project/outputs/yolo26n_tools/inference_example


In [13]:
image_rgb = cv2.cvtColor(
    image_bgr,
    cv2.COLOR_BGR2RGB,
)

overlay_rgb = cv2.cvtColor(
    overlay,
    cv2.COLOR_BGR2RGB,
)

plt.figure(figsize=(16, 5))

plt.subplot(1, 3, 1)
plt.imshow(image_rgb)
plt.title("Input")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(
    tool_mask,
    cmap="gray",
)
plt.title("YOLO26 Tool Mask")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(overlay_rgb)
plt.title("YOLO26 Overlay")
plt.axis("off")

plt.tight_layout()
plt.show()

<Figure size 1600x500 with 3 Axes>

## DepthAnything v2

In [14]:
import numpy as np
import cv2
import torch
import matplotlib.pyplot as plt

from PIL import Image
from transformers import pipeline

In [15]:
cfg["depth"] = {
    "enabled": True,
    "encoder": "vits",

    # Quantile locale calcolato dentro la tool mask
    "quantile": 0.25,

    "min_area": 20,

    # True: seleziona valori depth più bassi
    # False: seleziona valori depth più alti
    "use_low_depth": True,
}

In [16]:
class DepthAnythingV2Stage:
    def __init__(
        self,
        enabled=True,
        encoder="vits",
        checkpoint_path=None,
        device=None,
    ):
        self.enabled = enabled
        self.encoder = encoder
        self.checkpoint_path = checkpoint_path

        if device is None:
            self.device = torch.device(
                "cuda"
                if torch.cuda.is_available()
                else "cpu"
            )

        elif isinstance(device, torch.device):
            self.device = device

        elif device == 0:
            self.device = torch.device(
                "cuda"
            )

        else:
            self.device = torch.device(
                device
            )

        self.pipe = None

        if self.enabled:
            self._load_model()

    def _load_model(self):
        model_map = {
            "vits": (
                "depth-anything/"
                "Depth-Anything-V2-Small-hf"
            ),
            "vitb": (
                "depth-anything/"
                "Depth-Anything-V2-base-hf"
            ),
            "vitl": (
                "depth-anything/"
                "Depth-Anything-V2-Large-hf"
            ),
        }

        if self.encoder not in model_map:
            raise ValueError(
                f"Encoder non supportato: "
                f"{self.encoder}. "
                "Usa vits, vitb o vitl."
            )

        pipeline_device = (
            0
            if self.device.type == "cuda"
            else -1
        )

        self.pipe = pipeline(
            task="depth-estimation",
            model=model_map[self.encoder],
            device=pipeline_device,
        )

    @torch.no_grad()
    def infer(self, image_bgr):
        height, width = image_bgr.shape[:2]

        if not self.enabled:
            return np.zeros(
                (height, width),
                dtype=np.float32,
            )

        image_rgb = cv2.cvtColor(
            image_bgr,
            cv2.COLOR_BGR2RGB,
        )

        image_pil = Image.fromarray(
            image_rgb
        )

        output = self.pipe(image_pil)
        depth = output["depth"]

        if isinstance(depth, Image.Image):
            depth = np.asarray(depth)

        depth = np.asarray(
            depth,
            dtype=np.float32,
        )

        if depth.ndim == 3:
            depth = depth[..., 0]

        if depth.shape != (height, width):
            depth = cv2.resize(
                depth,
                (width, height),
                interpolation=cv2.INTER_LINEAR,
            )

        return depth

In [17]:
def normalize_depth(depth_map):
    depth_map = np.asarray(
        depth_map,
        dtype=np.float32,
    )

    depth_min = float(depth_map.min())
    depth_max = float(depth_map.max())

    if depth_max - depth_min < 1e-8:
        return np.zeros_like(depth_map)

    return (
        (depth_map - depth_min)
        / (depth_max - depth_min)
    )

In [18]:
depth_stage = DepthAnythingV2Stage(
    enabled=cfg["depth"]["enabled"],
    encoder=cfg["depth"]["encoder"],
    device=YOLO_DEVICE,
)

depth_map = depth_stage.infer(
    image_bgr
)

depth_normalized = normalize_depth(
    depth_map
)

image_rgb = cv2.cvtColor(
    image_bgr,
    cv2.COLOR_BGR2RGB,
)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(image_rgb)
plt.title("Input")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(
    depth_normalized,
    cmap="plasma",
)
plt.title("Depth Anything v2")
plt.axis("off")
plt.colorbar()

plt.tight_layout()
plt.show()

Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]

<Figure size 1200x500 with 3 Axes>

In [19]:
class FusionStage:
    def __init__(
        self,
        depth_quantile=0.25,
        min_area=20,
        use_low_depth=True,
    ):
        self.depth_quantile = depth_quantile
        self.min_area = min_area
        self.use_low_depth = use_low_depth

    def _remove_small_components(
        self,
        mask,
    ):
        if self.min_area <= 0:
            return mask

        binary = (
            mask > 0
        ).astype(np.uint8)

        num_labels, labels, stats, _ = (
            cv2.connectedComponentsWithStats(
                binary,
                connectivity=8,
            )
        )

        filtered = np.zeros_like(mask)

        for label_id in range(
            1,
            num_labels,
        ):
            area = stats[
                label_id,
                cv2.CC_STAT_AREA,
            ]

            if area >= self.min_area:
                filtered[
                    labels == label_id
                ] = 255

        return filtered

    def fuse(
        self,
        tool_mask,
        depth_map,
        debug=False,
    ):
        depth_norm = normalize_depth(
            depth_map
        )

        tool_region = tool_mask > 0

        tool_depth_values = (
            depth_norm[tool_region]
        )

        if tool_depth_values.size == 0:
            if debug:
                print(
                    "[Fusion] Tool mask vuota."
                )

            return np.zeros_like(
                tool_mask,
                dtype=np.uint8,
            )

        depth_cutoff = np.quantile(
            tool_depth_values,
            self.depth_quantile,
        )

        if self.use_low_depth:
            depth_region = (
                depth_norm <= depth_cutoff
            )
        else:
            depth_region = (
                depth_norm >= depth_cutoff
            )

        contact_mask = (
            tool_region & depth_region
        ).astype(np.uint8) * 255

        if contact_mask.any():
            contact_mask = cv2.medianBlur(
                contact_mask,
                5,
            )

        contact_mask = (
            self._remove_small_components(
                contact_mask
            )
        )

        if debug:
            print(
                "[Fusion] Global depth min:",
                float(depth_norm.min()),
            )

            print(
                "[Fusion] Global depth max:",
                float(depth_norm.max()),
            )

            print(
                "[Fusion] Tool depth min:",
                float(tool_depth_values.min()),
            )

            print(
                "[Fusion] Tool depth max:",
                float(tool_depth_values.max()),
            )

            print(
                "[Fusion] Tool depth mean:",
                float(tool_depth_values.mean()),
            )

            print(
                "[Fusion] Depth cutoff:",
                float(depth_cutoff),
            )

            print(
                "[Fusion] Quantile:",
                self.depth_quantile,
            )

            print(
                "[Fusion] Tool coverage:",
                float(tool_region.mean()),
            )

            print(
                "[Fusion] Contact coverage:",
                float(
                    (contact_mask > 0).mean()
                ),
            )

        return contact_mask

In [20]:
fusion_stage = FusionStage(
    depth_quantile=cfg["depth"]["quantile"],
    min_area=cfg["depth"]["min_area"],
    use_low_depth=cfg["depth"]["use_low_depth"],
)

In [21]:
instances = predict_instances(
    model=best_model,
    image_bgr=image_bgr,
    confidence=cfg["confidence"],
    device=YOLO_DEVICE,
)

yolo_tool_mask = merge_tool_masks(
    instances=instances,
    image_shape=image_bgr.shape,
)

depth_map = depth_stage.infer(
    image_bgr
)

depth_normalized = normalize_depth(
    depth_map
)

yolo_contact_mask = fusion_stage.fuse(
    tool_mask=yolo_tool_mask,
    depth_map=depth_map,
    debug=True,
)

[Fusion] Global depth min: 0.0
[Fusion] Global depth max: 1.0
[Fusion] Tool depth min: 0.2549019753932953
[Fusion] Tool depth max: 1.0
[Fusion] Tool depth mean: 0.8657426834106445
[Fusion] Depth cutoff: 0.8313725590705872
[Fusion] Quantile: 0.25
[Fusion] Tool coverage: 0.07121487258871159
[Fusion] Contact coverage: 0.017776223174829985


In [22]:
save_image(
    inference_dir / "depth_normalized.png",
    (
        depth_normalized * 255
    ).clip(0, 255).astype(np.uint8),
)

save_image(
    inference_dir / "contact_mask.png",
    yolo_contact_mask,
)

In [23]:
plt.figure(figsize=(16, 5))

plt.subplot(1, 4, 1)
plt.imshow(image_rgb)
plt.title("Input")
plt.axis("off")

plt.subplot(1, 4, 2)
plt.imshow(
    yolo_tool_mask,
    cmap="gray",
)
plt.title("YOLO tool mask aggregata")
plt.axis("off")

plt.subplot(1, 4, 3)
plt.imshow(
    depth_normalized,
    cmap="plasma",
)
plt.title("Depth Anything v2")
plt.axis("off")

plt.subplot(1, 4, 4)
plt.imshow(
    yolo_contact_mask,
    cmap="gray",
)
plt.title("YOLO + Depth contact")
plt.axis("off")

plt.tight_layout()
plt.show()

<Figure size 1600x500 with 4 Axes>

In [24]:
tool_pixels = (
    yolo_tool_mask > 0
)

tool_depth_values = (
    depth_normalized[tool_pixels]
)

print(
    "Tool pixels:",
    tool_depth_values.size,
)

if tool_depth_values.size > 0:
    percentiles = np.percentile(
        tool_depth_values,
        [
            1,
            5,
            10,
            25,
            50,
            75,
            90,
            95,
            99,
        ],
    )

    print(
        "Depth nella tool mask:"
    )

    print(
        "min:",
        float(tool_depth_values.min()),
    )

    print(
        "max:",
        float(tool_depth_values.max()),
    )

    print(
        "mean:",
        float(tool_depth_values.mean()),
    )

    print(
        "percentili:",
        percentiles,
    )

Tool pixels: 86121
Depth nella tool mask:
min: 0.2549019753932953
max: 1.0
mean: 0.8657426834106445
percentili: [    0.32157     0.70588     0.76863     0.83137      0.8902     0.93333     0.96471     0.97647     0.98824]


In [25]:
inference_dir = (
    YOLO_OUTPUT_DIR
    / "yolo_multiclass_inference"
)

inference_dir.mkdir(
    parents=True,
    exist_ok=True,
)

save_image(
    inference_dir / "depth_normalized.png",
    (
        depth_normalized * 255
    ).clip(0, 255).astype(np.uint8),
)

save_image(
    inference_dir / "yolo_tool_mask.png",
    yolo_tool_mask,
)

save_image(
    inference_dir / "contact_mask.png",
    yolo_contact_mask,
)

In [26]:
plt.figure(figsize=(16, 5))

plt.subplot(1, 4, 1)
plt.imshow(image_rgb)
plt.title("Input")
plt.axis("off")

plt.subplot(1, 4, 2)
plt.imshow(
    yolo_tool_mask,
    cmap="gray",
)
plt.title("YOLO tool mask aggregata")
plt.axis("off")

plt.subplot(1, 4, 3)
plt.imshow(
    depth_normalized,
    cmap="plasma",
)
plt.title("Depth Anything v2")
plt.axis("off")

plt.subplot(1, 4, 4)
plt.imshow(
    yolo_contact_mask,
    cmap="gray",
)
plt.title("YOLO + Depth contact candidate")
plt.axis("off")

plt.tight_layout()
plt.show()

<Figure size 1600x500 with 4 Axes>

In [27]:
fusion_low = FusionStage(
    depth_quantile=0.25,
    min_area=20,
    use_low_depth=True,
)

fusion_high = FusionStage(
    depth_quantile=0.25,
    min_area=20,
    use_low_depth=False,
)

contact_low = fusion_low.fuse(
    tool_mask=yolo_tool_mask,
    depth_map=depth_map,
    debug=True,
)

contact_high = fusion_high.fuse(
    tool_mask=yolo_tool_mask,
    depth_map=depth_map,
    debug=True,
)

[Fusion] Global depth min: 0.0
[Fusion] Global depth max: 1.0
[Fusion] Tool depth min: 0.2549019753932953
[Fusion] Tool depth max: 1.0
[Fusion] Tool depth mean: 0.8657426834106445
[Fusion] Depth cutoff: 0.8313725590705872
[Fusion] Quantile: 0.25
[Fusion] Tool coverage: 0.07121487258871159
[Fusion] Contact coverage: 0.017776223174829985
[Fusion] Global depth min: 0.0
[Fusion] Global depth max: 1.0
[Fusion] Tool depth min: 0.2549019753932953
[Fusion] Tool depth max: 1.0
[Fusion] Tool depth mean: 0.8657426834106445
[Fusion] Depth cutoff: 0.8313725590705872
[Fusion] Quantile: 0.25
[Fusion] Tool coverage: 0.07121487258871159
[Fusion] Contact coverage: 0.05425481596147231


In [28]:
plt.figure(figsize=(16, 5))

plt.subplot(1, 4, 1)
plt.imshow(image_rgb)
plt.title("Input")
plt.axis("off")

plt.subplot(1, 4, 2)
plt.imshow(
    depth_normalized,
    cmap="plasma",
)
plt.title("Depth")
plt.axis("off")

plt.subplot(1, 4, 3)
plt.imshow(
    contact_low,
    cmap="gray",
)
plt.title("Bottom 25% depth")
plt.axis("off")

plt.subplot(1, 4, 4)
plt.imshow(
    contact_high,
    cmap="gray",
)
plt.title("Top 25% depth")
plt.axis("off")

plt.tight_layout()
plt.show()

<Figure size 1600x500 with 4 Axes>